# 条件 Flow Matching（与论文 CFM 目标一致）+ CIFAR-10

参考 **Flow Matching for Generative Modeling**（Lipman et al., ICLR 2023）中的 **Conditional Flow Matching**：

1. 源分布 `p_0`：标准高斯，与数据同维。  
2. 目标：数据 `x_1 ~ p_1`（CIFAR-10 图像），**条件**为类别 `y`。  
3. **仿射插值路径**（论文 OT 路径）：`x_t = (1 - t) x_0 + t x_1`，`t ~ U(0,1)`。  
4. **目标速度场**（该路径上为常向量）：`u = x_1 - x_0`。  
5. **损失**（MSE，与式 (22)/(23) 类 CFM 目标一致）：
   `L = E[ || v_θ(x_t, t, y) - (x_1 - x_0) ||² ]`。

6. **采样（推理）**：`x_0 ∼ N(0,I)`，固定 `y`，用 ODE `dx/dt = v_θ(x,t,y)` 从 `t=0` 积到 `t=1`（欧拉法；可换更高阶积分器）。

数据集：**CIFAR-10** 由 `torchvision` **自动下载** 到 `data/cifar10/`。

In [ ]:
import os
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. 超参数与路径

In [ ]:
DATA_ROOT = os.path.abspath(os.path.join("data", "cifar10"))

IMAGE_SIZE = 32
IN_CHANNELS = 3
NUM_CLASSES = 10

PATCH_SIZE = 4
HIDDEN_SIZE = 384
DEPTH = 6
NUM_HEADS = 6
MLP_RATIO = 4.0

BATCH_SIZE = 128 if torch.cuda.is_available() else 64
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
LR = 2e-4
WEIGHT_DECAY = 0.0
EPOCHS = 50

N_STEPS_ODE = 100  # 推理时欧拉步数
SEED = 2023
CKPT_PATH = "flow_matching_cifar10_dit.pt"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. 数据：CIFAR-10 自动下载，图像归一化到 `[-1,1]`

In [ ]:
def get_cifar10_loaders(data_root, batch_size, num_workers):
    tfm = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    )
    train_set = datasets.CIFAR10(
        root=data_root, train=True, download=True, transform=tfm
    )
    test_set = datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=tfm
    )
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, test_loader, train_set.classes


train_loader, test_loader, class_names = get_cifar10_loaders(
    DATA_ROOT, BATCH_SIZE, NUM_WORKERS
)
print("classes:", class_names)
print("train batches:", len(train_loader), "test batches:", len(test_loader))

## 3. DiT 条件速度网络（与 Peebles & Xie DiT 块结构一致：adaLN-Zero）

In [ ]:
def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.frequency_embedding_size = frequency_embedding_size
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=t.device) / half
        )
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return emb

    def forward(self, t):
        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)
        return self.mlp(t_freq)


class LabelEmbedder(nn.Module):
    def __init__(self, num_classes, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, hidden_size)

    def forward(self, labels):
        return self.embedding(labels)


class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(
            hidden_size, num_heads, dropout=0.0, batch_first=True
        )
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        mlp_hidden = int(hidden_size * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, hidden_size),
        )
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(
            c
        ).chunk(6, dim=1)
        _x = modulate(self.norm1(x), shift_msa, scale_msa)
        attn_out, _ = self.attn(_x, _x, _x, need_weights=False)
        x = x + gate_msa.unsqueeze(1) * attn_out
        _x = modulate(self.norm2(x), shift_mlp, scale_mlp)
        x = x + gate_mlp.unsqueeze(1) * self.mlp(_x)
        return x


class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(), nn.Linear(hidden_size, 2 * hidden_size, bias=True)
        )
        nn.init.constant_(self.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.adaLN_modulation[-1].bias, 0)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        return self.linear(x)


def patchify(x, patch_size):
    B, C, H, W = x.shape
    h, w = H // patch_size, W // patch_size
    x = x.reshape(B, C, h, patch_size, w, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5).reshape(B, h * w, C * patch_size * patch_size)
    return x


def unpatchify(x, C, patch_size, h, w):
    B = x.shape[0]
    p = patch_size
    x = x.reshape(B, h, w, C, p, p)
    x = x.permute(0, 3, 1, 4, 2, 5).reshape(B, C, h * p, w * p)
    return x


class DiTConditionalVelocity(nn.Module):
    """v_θ(x, t, y)，输出与 x 同形状 (B,3,32,32)。"""

    def __init__(
        self,
        input_size=32,
        patch_size=4,
        in_channels=3,
        hidden_size=384,
        depth=6,
        num_heads=6,
        mlp_ratio=4.0,
        num_classes=10,
    ):
        super().__init__()
        self.input_size = input_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.hidden_size = hidden_size
        self.num_patches = (input_size // patch_size) ** 2
        patch_dim = patch_size * patch_size * in_channels

        self.x_embedder = nn.Linear(patch_dim, hidden_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, hidden_size))
        nn.init.normal_(self.pos_embed, std=0.02)

        self.t_embedder = TimestepEmbedder(hidden_size)
        self.y_embedder = LabelEmbedder(num_classes, hidden_size)

        self.blocks = nn.ModuleList(
            [DiTBlock(hidden_size, num_heads, mlp_ratio) for _ in range(depth)]
        )
        self.final_layer = FinalLayer(hidden_size, patch_size, in_channels)

    def forward(self, x, t, y):
        if t.dim() > 1:
            t = t.squeeze(-1)
        B, _, H, W = x.shape
        h = w = H // self.patch_size
        x = patchify(x, self.patch_size)
        x = self.x_embedder(x)
        x = x + self.pos_embed
        c = self.t_embedder(t) + self.y_embedder(y)
        for blk in self.blocks:
            x = blk(x, c)
        x = self.final_layer(x, c)
        x = unpatchify(x, self.in_channels, self.patch_size, h, w)
        return x

## 4. Flow Matching 损失（论文 CFM：线性路径 + 目标 `x_1 - x_0`）

In [ ]:
def flow_matching_loss(model, x1, y):
    """
    x1: 真实图像 (B,3,H,W)，已在 [-1,1]
    y: 类别 (B,) long
    """
    b = x1.shape[0]
    dev = x1.device
    x0 = torch.randn_like(x1)
    t = torch.rand(b, device=dev)
    tt = t.view(-1, 1, 1, 1)
    x_t = (1.0 - tt) * x0 + tt * x1
    target_velocity = x1 - x0
    pred = model(x_t, t, y)
    return F.mse_loss(pred, target_velocity)


@torch.no_grad()
def evaluate_loss(model, data_loader, device):
    model.eval()
    total, n = 0.0, 0
    for x1, y in data_loader:
        x1 = x1.to(device)
        y = y.to(device)
        loss = flow_matching_loss(model, x1, y)
        total += loss.item() * x1.size(0)
        n += x1.size(0)
    return total / max(n, 1)

## 5. 训练函数

In [ ]:
def train_flow_matching(
    model,
    train_loader,
    device,
    epochs,
    lr,
    weight_decay,
    ckpt_path,
    class_names,
    test_loader=None,
):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_loss = float("inf")
    hist = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        n = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for x1, y in pbar:
            x1 = x1.to(device)
            y = y.to(device)
            loss = flow_matching_loss(model, x1, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * x1.size(0)
            n += x1.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        tr = running / n
        hist["train_loss"].append(tr)

        val_l = None
        if test_loader is not None:
            val_l = evaluate_loss(model, test_loader, device)
            hist["val_loss"].append(val_l)
            print(
                f"Epoch {epoch}: train_loss={tr:.5f}  val_loss={val_l:.5f}"
            )
        else:
            print(f"Epoch {epoch}: train_loss={tr:.5f}")

        save_loss = val_l if val_l is not None else tr
        if save_loss < best_loss:
            best_loss = save_loss
            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": epoch,
                    "train_loss": tr,
                    "val_loss": val_l,
                    "class_names": class_names,
                    "config": {
                        "image_size": IMAGE_SIZE,
                        "patch_size": PATCH_SIZE,
                        "hidden_size": HIDDEN_SIZE,
                        "depth": DEPTH,
                        "num_heads": NUM_HEADS,
                        "num_classes": NUM_CLASSES,
                    },
                },
                ckpt_path,
            )
            print(f"  -> saved best checkpoint to {ckpt_path}")

    return hist

## 6. 推理：条件采样（ODE，欧拉法）

In [ ]:
@torch.no_grad()
def sample_conditional_images(
    model,
    labels,
    image_shape,
    n_steps,
    device,
    clamp_output=True,
):
    """
    从 x_0 ~ N(0,I) 出发，沿 dx/dt = v(x,t,y) 积分到 t=1。
    labels: (B,) long，每张图一个类别。
    image_shape: (C, H, W)
    """
    model.eval()
    b = labels.shape[0]
    x = torch.randn(b, *image_shape, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((b,), i * dt, device=device, dtype=torch.float32)
        v = model(x, t, labels)
        x = x + v * dt
    if clamp_output:
        x = x.clamp(-1.0, 1.0)
    return x


def load_checkpoint(path, model, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    return ckpt


def show_images(tensor, nrow=8, title=""):
    """tensor (N,3,H,W) in [-1,1]"""
    x = tensor.cpu().detach()
    x = (x + 1) / 2.0
    x = x.clamp(0, 1)
    n = x.shape[0]
    ncol = min(nrow, n)
    nrow_vis = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow_vis, ncol, figsize=(ncol * 1.5, nrow_vis * 1.5))
    axes = axes.flatten() if n > 1 else [axes]
    for i in range(n):
        im = x[i].permute(1, 2, 0).numpy()
        axes[i].imshow(im)
        axes[i].axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 7. 构建模型并训练（运行此格开始训练）

In [ ]:
model = DiTConditionalVelocity(
    input_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=IN_CHANNELS,
    hidden_size=HIDDEN_SIZE,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    num_classes=NUM_CLASSES,
).to(device)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"parameters: {n_params:.2f} M")

history = train_flow_matching(
    model,
    train_loader,
    device,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    ckpt_path=CKPT_PATH,
    class_names=class_names,
    test_loader=test_loader,
)

## 8. 训练曲线

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(history["train_loss"], label="train")
if history["val_loss"]:
    plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("FM loss")
plt.legend()
plt.title("Conditional Flow Matching (CFM)")
plt.tight_layout()
plt.show()

## 9. 推理：按标签生成一整批图

In [ ]:
# 加载最佳权重（若上面已训练可省略；冷启动推理时取消注释）
# load_checkpoint(CKPT_PATH, model, device)

n_per_class = 2
labels = torch.cat([torch.full((n_per_class,), i, dtype=torch.long) for i in range(NUM_CLASSES)]).to(device)

gen = sample_conditional_images(
    model,
    labels,
    (IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE),
    n_steps=N_STEPS_ODE,
    device=device,
)

show_images(gen, nrow=NUM_CLASSES, title="Conditional FM samples per label")

---

**说明**：
- 损失形式与 FM 论文 **线性 OT 路径** 下的 **conditional** 向量场回归一致；网络结构采用 **DiT（Transformer）**，来自另一篇论文，仅作 `v_θ` 的参数化。
- CIFAR 上要较好效果通常需要 **更长训练 / 更大模型 / EMA**；可按需增大 `EPOCHS`、`HIDDEN_SIZE`、`DEPTH`。
- `DATA_ROOT` 指向 `data/cifar10`，首次运行会自动下载。